In [1]:
from pydrake.all import StartMeshcat

meshcat = StartMeshcat()

INFO:drake:Meshcat listening for connections at http://localhost:7000


In [4]:
# Import some basic libraries and functions for this tutorial.
import numpy as np
import os

from pydrake.common import temp_directory
from pydrake.geometry import SceneGraphConfig, StartMeshcat
from pydrake.math import RigidTransform, RollPitchYaw
from pydrake.multibody.parsing import Parser
from pydrake.multibody.plant import AddMultibodyPlantSceneGraph
from pydrake.systems.analysis import Simulator
from pydrake.systems.framework import DiagramBuilder
from pydrake.visualization import AddDefaultVisualization, ModelVisualizer

In [3]:
#Only KUKA arm visualization

# First we'll choose one of Drake's example model files, a KUKA iiwa arm.
iiwa7_model_url = (
    "package://drake_models/iiwa_description/sdf/"
    "iiwa7_with_box_collision.sdf")

# Create a model visualizer and add the robot arm.
visualizer = ModelVisualizer(meshcat=meshcat)
visualizer.parser().AddModels(url=iiwa7_model_url)

# When this notebook is run in test mode it needs to stop execution without
# user interaction. For interactive model visualization you won't normally
# need the 'loop_once' flag.
test_mode = True if "TEST_SRCDIR" in os.environ else False

# Start the interactive visualizer.
# Click the "Stop Running" button in MeshCat when you're finished.
visualizer.Run(loop_once=test_mode)

INFO:drake:Click 'Stop Running' or press Esc to quit


In [5]:
#Only a double pendulum visualization

from pydrake.common import FindResourceOrThrow

# Create a model visualizer and add the robot arm.
visualizer = ModelVisualizer(meshcat=meshcat)
visualizer.parser().AddModels(
    FindResourceOrThrow("drake/examples/pendulum/Pendulum.urdf"))

# When this notebook is run in test mode it needs to stop execution without
# user interaction. For interactive model visualization you won't normally
# need the 'loop_once' flag.
test_mode = True if "TEST_SRCDIR" in os.environ else False

# Start the interactive visualizer.
# Click the "Stop Running" button in MeshCat when you're finished.
visualizer.Run(loop_once=test_mode)

INFO:drake:Click 'Stop Running' or press Esc to quit


In [48]:
from pydrake.multibody.plant import CoulombFriction
from pydrake.math import RotationMatrix
from pydrake.geometry import HalfSpace

def AddGround(plant):
    """
    Add a flat ground with friction
    """

    # Constants
    transparent_color = np.array([0.5,0.5,0.5,0])
    nontransparent_color = np.array([0.5,0.5,0.5,0.1])

    p_GroundOrigin = [0, 0.0, 0.0]
    R_GroundOrigin = RotationMatrix.MakeXRotation(0.0)
    X_GroundOrigin = RigidTransform(R_GroundOrigin,p_GroundOrigin)

    # Set Up Ground on Plant

    surface_friction = CoulombFriction(
            static_friction = 0.7,
            dynamic_friction = 0.5)
    plant.RegisterCollisionGeometry(
            plant.world_body(),
            X_GroundOrigin,
            HalfSpace(),
            "ground_collision",
            surface_friction)
    plant.RegisterVisualGeometry(
      plant.world_body(),
      X_GroundOrigin,
      HalfSpace(),
      "ground_visual",
      nontransparent_color)

In [ ]:
def create_scene(obj_url, sim_time_step=0.0):
    # Create a diagram builder and add a scene graph.
    builder = DiagramBuilder()
    # Create plant and scene_graph from Diagram
    plant, scene_graph = AddMultibodyPlantSceneGraph(
        builder, time_step=sim_time_step)
    # Adding the model to the parser
    parser = Parser(builder)
    # parser.AddModels(url=obj_url)
    parser.AddModels(file_name=obj_url)    

    # Add a flat ground to the simulation
    AddGround(plant=plant)
    
    plant.Finalize()
    plant_context = plant.CreateDefaultContext()

    # Start the robot at frame z=0.5
    plant.SetDefaultFreeBodyPose(
      plant.GetBodyByName("base_link"),
      RigidTransform([0, 0, 0.5])
    )

    AddDefaultVisualization(builder=builder, meshcat=meshcat)
    
    return builder.Build()

In [32]:
pendulum_model_url = ("package://drake/examples/pendulum/Pendulum.urdf")
robot_model_url = ("robot.urdf")

In [62]:
def initialize_simulation(diagram):
    simulator = Simulator(diagram)
    simulator.Initialize()
    simulator.set_target_realtime_rate(1.)
    return simulator

def run_simulation(sim_time_step):
    # Add a double pendulum to the scene.
    print(f'Test mode is {test_mode}')
    diagram = create_scene(robot_model_url, sim_time_step)
    simulator = initialize_simulation(diagram)
    meshcat.StartRecording()
    finish_time = 0.1 if test_mode else 10.0
    simulator.AdvanceTo(finish_time)
    meshcat.PublishRecording()

# Run the simulation with a small time step. Try gradually increasing it!
run_simulation(sim_time_step=0.001)


Test mode is False
